# Bayan wake-word — Colab trainer (faithful port of openWakeWord automatic training)

Based on openWakeWord's `automatic_model_training.ipynb`, but with the two rot points that broke
earlier runs fixed at the root and everything **pinned**:
- Piper: uses the openWakeWord author's **fork** `dscripka/piper-sample-generator` (pinned commit),
  which provides the root `generate_samples.py` that `train.py` imports — the current rhasspy repo
  was repackaged and no longer has it (that caused `ModuleNotFoundError: generate_samples`).
- **No TensorFlow**: `train.py` needs it only for the optional `--convert_to_tflite` step (never
  called here). We ship ONNX, so the un-installable `tensorflow-cpu==2.8.1` pin is removed.
- openWakeWord is a single pinned source (clone + editable), never mixed with the pip package.

It adds only: upload your ZIP, inject your real clips into the positive set before generation, and
export `bayan_wake.onnx` + shared feature models + `wake_meta.json` + a downloadable ZIP.

## Run in TWO phases (do NOT skip phase 1)
1. **Pre-flight first.** Runtime → Change runtime type → **GPU (T4)**. Then run only the first two
   cells: **Setup** (installs, ~3–5 min) and **PRE-FLIGHT** (~2–3 min). Pre-flight verifies the whole
   fragile chain — pinned commits, `generate_samples` import, `torch_audiomentations`, the Piper model
   loading, one real synthetic "Bayan" WAV, `train.py` initialising, and ONNX export. If it prints
   `ALL PRE-FLIGHT CHECKS PASSED`, continue. If it fails, it STOPS — do not run the long cells.
2. **Full run.** Only after pre-flight passes: **Runtime → Run all**, and upload
   `wake_data_v1-egyptian-pilot.zip` when asked. No code edits, no manual installs.

> openWakeWord automatic training is **Linux-only** (Piper) — Colab is Linux, so this is fine.
> Full run ≈ 1–1.5 h on a free T4 (data download + synthetic generation + training).

In [ ]:
## Environment setup — PINNED, single-source, matched Piper fork, no TensorFlow (ONNX only).

# System library for the Piper phonemizer used by the sample generator fork
!apt-get -qq update && apt-get -qq install -y espeak-ng

# piper-sample-generator: the openWakeWord AUTHOR'S fork — it has the root generate_samples.py that
# train.py imports (`from generate_samples import generate_samples`). Pinned so the API can't drift.
# Do NOT use rhasspy's latest: it was restructured into a `piper_sample_generator` package with no
# root module, which is exactly what caused `ModuleNotFoundError: No module named 'generate_samples'`.
!git clone https://github.com/dscripka/piper-sample-generator
!cd piper-sample-generator && git checkout f1988a4d54eddb23d99e86f0adfef6226a85acc7
!mkdir -p piper-sample-generator/models
!wget -O piper-sample-generator/models/en-us-libritts-high.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v1.0.0/en-us-libritts-high.pt'
!pip install espeak-phonemizer webrtcvad

# openwakeword — SINGLE source (clone + editable), pinned to a commit that matches the fork above.
# (Do NOT also `pip install openwakeword` — mixing the two was another failure.)
!git clone https://github.com/dscripka/openwakeword
!cd openwakeword && git checkout 368c03716d1e92591906a84949bc477f3a834455
!pip install -e ./openwakeword

# Training dependencies (pinned; includes torchinfo + speechbrain). TensorFlow / tensorflow_probability
# / onnx_tf are INTENTIONALLY OMITTED: train.py imports them only inside the optional --convert_to_tflite
# path, which we never call. The shipped model is ONNX (torch.onnx), so dropping the TF stack removes a
# major version-rot risk (tensorflow-cpu==2.8.1 has no wheels on modern Colab Python).
# torch-audiomentations 0.12.0 (NOT 0.11.0): 0.11.0 calls torchaudio.set_audio_backend(), removed in
# torchaudio>=2.1; 0.12.0 dropped that call and imports cleanly on modern torchaudio.
# onnxscript is required by torch's ONNX exporter on current PyTorch.
!pip install mutagen==1.47.0 torchinfo==1.8.0 torchmetrics==1.2.0 speechbrain==0.5.14 audiomentations==0.33.0 torch-audiomentations==0.12.0 acoustics==0.2.6 pronouncing==0.2.0 datasets==2.14.6 deep-phonemizer==0.0.19 onnxscript soundfile

# Required feature models (used by openWakeWord's feature extractor + the browser inference chain)
import os
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite

# ---- Compatibility patches for modern PyTorch (edit installed sources so they hold in the train.py subprocess too) ----
import pathlib
# (2) The Piper checkpoint is the openWakeWord author's TRUSTED release; torch>=2.6 defaults
#     torch.load(weights_only=True), which refuses its pickled SynthesizerTrn. Load it fully.
_gs = pathlib.Path("piper-sample-generator/generate_samples.py")
_s = _gs.read_text()
if "torch.load(model_path)" in _s and "weights_only" not in _s:
    _gs.write_text(_s.replace("torch.load(model_path)", "torch.load(model_path, weights_only=False)"))
    print("patched piper generate_samples.py -> weights_only=False")
# (4) Force openWakeWord's ONNX export to the stable (legacy) exporter. The new dynamo default needs
#     onnxscript and is stricter; opset-13 legacy export is what this model was built for.
_tp = pathlib.Path("openwakeword/openwakeword/train.py")
_t = _tp.read_text()
if "opset_version=13)" in _t and "dynamo=False" not in _t:
    _tp.write_text(_t.replace("opset_version=13)", "opset_version=13, dynamo=False)"))
    print("patched openwakeword/train.py -> torch.onnx.export(..., dynamo=False)")
# (5) torch-audiomentations get_audio_metadata() calls torchaudio.info(), removed in current torchaudio;
#     AddBackgroundNoise depends on it. Read the same (num_frames, sample_rate) via soundfile instead.
import importlib.util as _ilu
_ta = _ilu.find_spec("torch_audiomentations")
if _ta and _ta.submodule_search_locations:
    _io = pathlib.Path(list(_ta.submodule_search_locations)[0]) / "utils" / "io.py"
    _s = _io.read_text()
    _old = "info = torchaudio.info(str(file_path))"
    if _old in _s:
        _io.write_text(_s.replace(_old,
            'import soundfile as _sf; _si = _sf.info(str(file_path)); '
            'info = type("MD", (), {"num_frames": _si.frames, "sample_rate": _si.samplerate})()'))
        print("patched torch_audiomentations io.py -> torchaudio.info replaced with soundfile.info")
print("compatibility patches applied.")

In [ ]:
# ================= PRE-FLIGHT (run this right after Setup) =================
# Fast checks (~2-3 min). If ANY check fails, this cell raises and STOPS — do NOT run the
# data-download / generate / augment / train cells until every check passes.
import os, sys, subprocess, glob, tempfile
PIN_OWW   = "368c03716d1e92591906a84949bc477f3a834455"
PIN_PIPER = "f1988a4d54eddb23d99e86f0adfef6226a85acc7"
PIPER_DIR = os.path.abspath("piper-sample-generator")
PF_WAV    = "_preflight_bayan_16k.wav"
OK, FAIL = [], []
def check(name, fn):
    try:
        fn(); OK.append(name); print("  PASS :", name)
    except Exception as e:
        FAIL.append((name, repr(e))); print("  FAIL :", name, "->", repr(e))

def c1_oww_commit():
    sha = subprocess.check_output(["git","-C","openwakeword","rev-parse","HEAD"]).decode().strip()
    assert sha == PIN_OWW, "openwakeword HEAD " + sha + " != pinned " + PIN_OWW

def c2_piper_commit():
    sha = subprocess.check_output(["git","-C","piper-sample-generator","rev-parse","HEAD"]).decode().strip()
    assert sha == PIN_PIPER, "piper fork HEAD " + sha + " != pinned " + PIN_PIPER

def c3_generate_samples_import():
    if PIPER_DIR not in sys.path: sys.path.insert(0, PIPER_DIR)
    from generate_samples import generate_samples  # noqa: F401

def c4_torch_audiomentations():
    import torch_audiomentations  # noqa: F401

def c5_piper_model_present():
    p = "piper-sample-generator/models/en-us-libritts-high.pt"
    assert os.path.exists(p) and os.path.getsize(p) > 1_000_000, "model missing/too small: " + p

def c6_generate_and_resample_16k():
    import numpy as np, math, scipy.io.wavfile as _wf
    from scipy.signal import resample_poly
    if PIPER_DIR not in sys.path: sys.path.insert(0, PIPER_DIR)
    from generate_samples import generate_samples
    d = tempfile.mkdtemp()
    generate_samples(text=["bayan"], output_dir=d, max_samples=1, batch_size=1,
                     file_names=["s.wav"], auto_reduce_batch_size=True)
    raw = glob.glob(os.path.join(d, "*.wav"))
    assert raw and os.path.getsize(raw[0]) > 1000, "no synthetic WAV produced"
    sr, data = _wf.read(raw[0]); print("         raw Piper sr:", sr, "Hz")
    x = data.astype(np.float32) / (32768.0 if data.dtype == np.int16 else 1.0)
    if x.ndim > 1: x = x.mean(axis=1)
    if sr != 16000:
        g = math.gcd(int(sr), 16000); x = resample_poly(x, 16000 // g, int(sr) // g)
    _wf.write(PF_WAV, 16000, (np.clip(x, -1, 1) * 32767).astype(np.int16))
    rsr, _ = _wf.read(PF_WAV)
    assert rsr == 16000, "resample to 16 kHz failed, got " + str(rsr)
    print("         synthetic WAV resampled ->", rsr, "Hz  (saved " + PF_WAV + ")")

def c7_train_initializes():
    # `--help` forces Python to import every top-level dependency of train.py (torch, torchinfo,
    # torchmetrics, speechbrain via openwakeword.data, AudioFeatures, ...) and build the arg parser.
    # A missing module (the class of error you hit) fails here in seconds, not hours in.
    r = subprocess.run([sys.executable, "openwakeword/openwakeword/train.py", "--help"],
                       capture_output=True, text=True)
    assert r.returncode == 0, "train.py --help failed:\n" + (r.stderr[-1500:] or r.stdout[-1500:])

def c8_onnx_export():
    import onnxscript  # noqa: F401  (ONNX exporter dependency; must be installed)
    import torch, torch.nn as nn
    m = nn.Sequential(nn.Linear(16, 8), nn.ReLU(), nn.Linear(8, 1))
    f = os.path.join(tempfile.mkdtemp(), "preflight.onnx")
    torch.onnx.export(m, torch.rand(1, 16), f, opset_version=13, dynamo=False)  # exact call train.py now uses
    assert os.path.getsize(f) > 0

def c9_feature_extraction():
    # A REAL feature-extraction pass (not just imports): runs openWakeWord's melspectrogram + embedding
    # ONNX models on the 16 kHz clip from c6 and checks a non-trivial embedding tensor comes out. This is
    # the exact chain --augment_clips uses, so it catches feature-stage breakage before the long run.
    import numpy as np, scipy.io.wavfile as _wf
    from openwakeword.utils import AudioFeatures
    sr, data = _wf.read(PF_WAV); assert sr == 16000, "preflight clip not 16 kHz"
    if data.ndim > 1: data = data[:, 0]
    x = data.astype(np.float32)
    x = np.pad(x, (0, 16000 - len(x))) if len(x) < 16000 else x[:16000]
    emb = AudioFeatures(device="cpu").embed_clips(np.array([x]), batch_size=1)
    assert emb.ndim == 3 and emb.shape[0] == 1 and emb.shape[1] > 0 and emb.shape[2] > 0, \
        "unexpected feature shape " + str(emb.shape)
    print("         real feature extraction OK, embedding shape:", emb.shape)

def c10_bg_metadata_path():
    # The exact metadata path AddBackgroundNoise uses. On current torchaudio this used to call the
    # removed torchaudio.info(); the setup patch routes it through soundfile. Must NOT raise here.
    from torch_audiomentations.utils.io import Audio
    ns, sr = Audio.get_audio_metadata(PF_WAV)
    assert ns > 0 and sr == 16000, "unexpected audio metadata " + str((ns, sr))
    print("         AddBackgroundNoise metadata path OK -> num_samples=" + str(ns) + " sr=" + str(sr))

for name, fn in [
    ("pinned openWakeWord commit", c1_oww_commit),
    ("pinned dscripka Piper fork commit", c2_piper_commit),
    ("generate_samples imports", c3_generate_samples_import),
    ("torch-audiomentations imports", c4_torch_audiomentations),
    ("Piper model present + non-trivial", c5_piper_model_present),
    ("synthetic 'Bayan' WAV generated + resampled to 16 kHz", c6_generate_and_resample_16k),
    ("train.py initializes (all imports + arg parser)", c7_train_initializes),
    ("ONNX export works (onnxscript + torch.onnx dynamo=False)", c8_onnx_export),
    ("real feature extraction (melspec + embedding on 16 kHz clip)", c9_feature_extraction),
    ("AddBackgroundNoise metadata path (soundfile, no torchaudio.info)", c10_bg_metadata_path),
]:
    check(name, fn)

print("\nPRE-FLIGHT:", len(OK), "passed,", len(FAIL), "failed")
assert not FAIL, ("PRE-FLIGHT FAILED — do NOT run the long cells. Fix these first:\n"
                  + "\n".join("- " + n + ": " + e for n, e in FAIL))
print("ALL PRE-FLIGHT CHECKS PASSED — safe to Runtime -> Run all for the full training.")

In [ ]:
import os, sys, math, glob, subprocess, uuid
import numpy as np
import torch
from pathlib import Path
import yaml
import datasets
import scipy
import scipy.io.wavfile as wf
from scipy.signal import resample_poly
from tqdm import tqdm

# --- Resampling helpers: Piper emits 22.05 kHz, but openWakeWord augment/feature-extraction REQUIRE
#     16 kHz mono PCM-16. These convert clips IN PLACE and are used right after synthetic generation.
def to_16k_mono_pcm16(path):
    sr, data = wf.read(path)
    if data.dtype == np.int16:   x = data.astype(np.float32) / 32768.0
    elif data.dtype == np.int32: x = data.astype(np.float32) / 2147483648.0
    elif data.dtype == np.uint8: x = (data.astype(np.float32) - 128) / 128.0
    else:                        x = data.astype(np.float32)
    if x.ndim > 1: x = x.mean(axis=1)                       # -> mono
    if sr != 16000:
        g = math.gcd(int(sr), 16000)
        x = resample_poly(x, 16000 // g, int(sr) // g)      # exact-ratio resample
    wf.write(path, 16000, (np.clip(x, -1.0, 1.0) * 32767).astype(np.int16))

def resample_tree_to_16k(dirs):
    n = 0
    for d in dirs:
        for p in glob.glob(os.path.join(d, "*.wav")):
            to_16k_mono_pcm16(p); n += 1
    bad = [p for d in dirs for p in glob.glob(os.path.join(d, "*.wav")) if wf.read(p)[0] != 16000]
    assert not bad, "NOT 16 kHz after resample: " + str(bad[:10])
    print("resampled to 16 kHz:", n, "clips across", len([d for d in dirs if os.path.isdir(d)]), "dirs")

In [ ]:
# Upload your recordings ZIP (from the recorder tool). Run-all will pause here for the file dialog.
from google.colab import files
import zipfile, glob
print("Choose wake_data_v1-egyptian-pilot.zip ...")
up = files.upload()
zname = [f for f in up if f.endswith(".zip")][0]
zipfile.ZipFile(zname).extractall(".")
real_clips = sorted(glob.glob("wake_data/positive/**/*.wav", recursive=True))
print("real positive recordings found:", len(real_clips))
assert len(real_clips) > 0, "No positives found under wake_data/positive in the uploaded ZIP."


In [ ]:
# Download room impulse responses collected by MIT
# https://mcdermottlab.mit.edu/Reverb/IR_Survey.html

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)

# Save clips to 16-bit PCM wav files
for row in tqdm(rir_dataset):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

In [ ]:
## Download noise and background audio (Audioset part + FMA small)

if not os.path.exists("audioset"):
    os.mkdir("audioset")

fname = "bal_train09.tar"
out_dir = f"audioset/{fname}"
link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
!wget -O {out_dir} {link}
!cd audioset && tar -xvf bal_train09.tar

output_dir = "./audioset_16k"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset):
    name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))

n_hours = 1  # recommend increasing for full-scale training
for i in tqdm(range(n_hours*3600//30)):  # FMA clips are 30 s each
    row = next(fma_dataset)
    name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
    i += 1
    if i == n_hours*3600//30:
        break

In [ ]:
# Download pre-computed openWakeWord features for training + validation
# training set (~2,000 hours, ACAV100M) and false-positive validation set (~11 hours)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

In [ ]:
# Load the OFFICIAL config template (has piper_sample_generator_path + all required keys), then override.
MODEL_VERSION   = "bayan-wake-v1-egyptian"
DATASET_VERSION = "v1-egyptian-pilot"

config = yaml.load(open("openwakeword/examples/custom_model.yml").read(), yaml.Loader)
config["target_phrase"] = ["bayan", "beyan", "bayaan"]   # English spellings Piper can voice; your real Arabic clips add the rest
config["model_name"]    = "bayan"
config["n_samples"]     = 5000      # synthetic positives (topped up on top of your real clips)
config["n_samples_val"] = 1000
config["steps"]         = 15000
config["target_accuracy"] = 0.6
config["target_recall"]   = 0.25
config["background_paths"] = ["./audioset_16k", "./fma"]
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}
# piper_sample_generator_path stays "./piper-sample-generator" (from the template) — matches the clone above.
with open("bayan.yaml", "w") as f:
    yaml.dump(config, f)
print("piper:", config["piper_sample_generator_path"], "| output_dir:", config["output_dir"], "| model:", config["model_name"])

In [ ]:
# Put your real clips into the positive dirs so generation tops up around them and augmentation includes them.
# (train.py counts existing files and only generates the remainder; --augment_clips globs every *.wav here.)
import shutil
base = os.path.join(config["output_dir"], config["model_name"])
ptr = os.path.join(base, "positive_train"); pte = os.path.join(base, "positive_test")
os.makedirs(ptr, exist_ok=True); os.makedirs(pte, exist_ok=True)

def place(src, dst, min_secs=1.5, sr=16000):
    r, dat = scipy.io.wavfile.read(src)
    if getattr(dat, "ndim", 1) > 1: dat = dat[:, 0]
    if r != sr:  # recorder already exports 16 kHz; this is just a safety net
        dat = np.interp(np.linspace(0, len(dat), int(len(dat)*sr/r), endpoint=False),
                        np.arange(len(dat)), dat)
    dat = dat.astype(np.int16)
    need = int(min_secs*sr)
    if len(dat) < need:  # pad short wake clips so the feature window is satisfied
        dat = np.concatenate([dat, np.zeros(need-len(dat), dtype=np.int16)])
    scipy.io.wavfile.write(dst, sr, dat)

for i, p in enumerate(real_clips):
    dst_dir = pte if (i % 6 == 0) else ptr   # hold ~1/6 out for validation
    place(p, os.path.join(dst_dir, f"real_{i:03d}.wav"))
print("real clips injected -> positive_train:", len(os.listdir(ptr)), "| positive_test:", len(os.listdir(pte)))

In [ ]:
# Step 1: generate synthetic clips (tops up to n_samples around your real clips). ~10-20 min on a T4.
subprocess.run([sys.executable, TRAIN, "--training_config", "bayan.yaml", "--generate_clips"], check=True)
assert glob.glob(BASE + "/positive_train/*.wav"), "Step 1 produced no positive_train clips"
assert glob.glob(BASE + "/negative_train/*.wav"), "Step 1 produced no negative_train clips"
print("Step 1 OK — positive_train:", len(glob.glob(BASE + "/positive_train/*.wav")),
      "negative_train:", len(glob.glob(BASE + "/negative_train/*.wav")))

In [ ]:
# Step 1b: RESAMPLE every generated clip to 16 kHz mono PCM-16 (Piper emits 22.05 kHz; openWakeWord
# augment + feature extraction REQUIRE 16 kHz). Covers synthetic positives AND the Piper-generated
# adversarial negatives. This is the step whose absence made Step 2 fail silently before.
resample_tree_to_16k(CLIP_DIRS)
bad = [p for d in CLIP_DIRS for p in glob.glob(d + "/*.wav") if wf.read(p)[0] != 16000]
assert not bad, "Some clips are still not 16 kHz: " + str(bad[:10])
print("Step 1b OK — all generated clips are 16 kHz")

In [ ]:
# Step 2: augment (real + synthetic) + compute features. check=True stops the notebook on any error.
subprocess.run([sys.executable, TRAIN, "--training_config", "bayan.yaml",
                "--augment_clips", "--overwrite"], check=True)
for f in FEATS:
    assert os.path.exists(f) and os.path.getsize(f) > 0, "missing/empty feature file: " + f
    assert np.load(f, mmap_mode="r").shape[0] > 0, "empty feature array: " + f
print("Step 2 OK — all four feature files present:", [os.path.basename(f) for f in FEATS])

In [ ]:
# Step 3: train. GUARD — refuse to start unless all four feature files exist (never train on partial data).
assert all(os.path.exists(f) and os.path.getsize(f) > 0 for f in FEATS), \
    "Refusing to train: Step 2 feature files are missing. Re-run Step 2 first."
subprocess.run([sys.executable, TRAIN, "--training_config", "bayan.yaml", "--train_model"], check=True)
_onnx = [p for p in glob.glob("my_custom_model/**/*.onnx", recursive=True) if "bayan" in os.path.basename(p).lower()]
assert _onnx, "Step 3 finished but produced no bayan*.onnx"
print("Step 3 OK — classifier ONNX:", max(_onnx, key=os.path.getsize))

In [ ]:
# Package: bayan_wake.onnx + the shared feature models + versioned wake_meta.json -> downloadable ZIP.
import shutil, json, hashlib, glob
cands = glob.glob(os.path.join(config["output_dir"], "**", "bayan*.onnx"), recursive=True)
assert cands, "No bayan*.onnx produced — read the Step-3 output above for the training error."
onnx_src = max(cands, key=os.path.getsize)
os.makedirs("bayan-wake-out", exist_ok=True)
shutil.copy(onnx_src, "bayan-wake-out/bayan_wake.onnx")
onnx_sha = hashlib.sha256(open("bayan-wake-out/bayan_wake.onnx", "rb").read()).hexdigest()

# openWakeWord inference is a CHAIN: melspectrogram.onnx -> embedding_model.onnx -> this classifier.
# Bundle the two shared feature models so the browser has the full pipeline.
res = "openwakeword/openwakeword/resources/models"
for m in ["melspectrogram.onnx", "embedding_model.onnx"]:
    if os.path.exists(os.path.join(res, m)): shutil.copy(os.path.join(res, m), "bayan-wake-out/"+m)

ds_sha = None
if os.path.exists("wake_data/recording_manifest.json"):
    ds_sha = hashlib.sha256(open("wake_data/recording_manifest.json", "rb").read()).hexdigest()

meta = {
  "modelVersion": MODEL_VERSION, "datasetVersion": DATASET_VERSION,
  "trainer": "openWakeWord automatic_model_training (faithful port)",
  "targetPhrase": config["target_phrase"], "wakePhrases": ["Bayan", "بيان", "يا بيان"],
  "sampleRate": 16000, "inferenceChain": ["melspectrogram.onnx", "embedding_model.onnx", "bayan_wake.onnx"],
  "onnxSource": os.path.relpath(onnx_src), "onnxSha256": onnx_sha, "datasetManifestSha256": ds_sha,
  "realPositives": len(real_clips), "syntheticPositives": config["n_samples"], "steps": config["steps"],
  # Fill from a real measurement (see the eval note below); never invent numbers:
  "metrics": {"quietDetection": None, "noisyDetection": None, "falseActivationsPerHour": None, "p95LatencyMs": None},
}
open("bayan-wake-out/wake_meta.json", "w").write(json.dumps(meta, ensure_ascii=False, indent=2))
print(json.dumps(meta, ensure_ascii=False, indent=2))

zip_name = f"bayan-wake-model_{MODEL_VERSION}"
shutil.make_archive(zip_name, "zip", "bayan-wake-out")
from google.colab import files as F
F.download(zip_name + ".zip")
print("DONE ->", zip_name + ".zip  (bayan_wake.onnx + melspectrogram.onnx + embedding_model.onnx + wake_meta.json)")

## After it finishes
You get `bayan-wake-model_bayan-wake-v1-egyptian.zip` containing `bayan_wake.onnx`,
the shared `melspectrogram.onnx` + `embedding_model.onnx`, and `wake_meta.json`. Send it back.

**Honest eval note (single-speaker pilot):** this v1 was trained on one speaker's 24 real clips plus
synthetic data. On-speaker detection will look high but does NOT prove cross-speaker performance, and
false-activations/hour must be MEASURED against real long negative audio — those numbers come from the
deployed benchmark, not from training. Leave `metrics` null until measured; never invent them.